In [ ]:
from dash_core_components import *
from dash_html_components import *
from dash import Dash
import dash.dependencies as dp
from json import load, dump


def mkTable(a):
    return Thead(Tr([Th("key"), Th("value")])), Tbody(
        [Tr([Td(i), Td(j)]) for i, j in a.items()]
    )


a = load(open("kv.json"))
app = Dash()
app.layout = Div(
    [
        Table(mkTable(a), "table"),
        Input("object"),
        Input("color"),
        Button("add", "button"),
    ]
)


@app.callback(
    dp.Output("table", "children"),
    dp.Input("button", "n_clicks"),
    dp.State("object", "value"),
    dp.State("color", "value"),
)
def _(_, key, value):
    a[key] = value
    dump(a, open("kv.json", "w"), ensure_ascii=0, separators=",:")
    return mkTable(a)


app.run_server()

In [ ]:
from dash_core_components import *
from dash_html_components import *
from dash import Dash
import dash.dependencies as dp
import plotly.express as ex
import numpy as np


def rank(a, k=0.4):
    a = k / a.sum(0) * a
    a += (1 - k) / len(a)
    a, b = np.linalg.eig(a)
    return np.abs(b[:, np.isclose(a, 1).nonzero()[0][0]].real)


a = np.full((16, 16), 1e-9, "f")
app = Dash()
app.layout = Div(
    [
        Graph("graph", figure=ex.bar(rank(a))),
        Input("i", "我"),
        Button("赞", "button"),
        Input("j", "谁"),
    ]
)


@app.callback(
    dp.Output("graph", "figure"),
    dp.Input("button", "n_clicks"),
    dp.State("i", "value"),
    dp.State("j", "value"),
)
def _(_, i, j):
    i = int(i)
    j = int(j)
    assert i != j
    a[j, i] += 1
    return ex.bar(rank(a))


app.run_server('localhost')